In [11]:
import pandas as pd

# 0. Read predictor-processed dataset

df_project = pd.read_csv(
    "../data/data_processed/df_predictors_processed.csv",
    low_memory=False
)

print("Predictor-processed shape:", df_project.shape)

Predictor-processed shape: (14512, 49)


In [12]:
# 1. Define final analysis variables

behaviour_vars = (
    [f"i12_health_{i}" for i in range(1, 9)] +
    [f"i12_health_{i}" for i in range(11, 21)]
)

employment_vars = [
    f"employment_status_{i}" for i in range(1, 8)
]

contact_vars = [
    "i1_health",
    "i2_health",
    "i7a_health"
]

feature_vars = [
    # 18 protective behaviour variables
    *behaviour_vars,

    # 2 demographic variables
    "gender",
    "age_num",

    # 2 household variables
    "household_size_num",
    "household_children_group",

    # 7 employment variables
    *employment_vars,

    # 3 contact variables
    *contact_vars,

    # 2 testing variables
    "personal_test_status",
    "household_test_status",

    # 1 symptom variable
    "covid_symptom_status",

    # 1 comorbidity variable
    "comorbidity_status",

    # 1 survey-wave variable
    "survey_wave"
]

analysis_vars = [
    # 2 outcome variables
    "feasibility_binary",
    "willingness_binary",

    # 37 predictor variables
    *feature_vars
]

print("\nNumber of final predictors:", len(feature_vars))
print("Number of final analysis variables:", len(analysis_vars))



Number of final predictors: 37
Number of final analysis variables: 39


In [21]:
excluded_from_final = [
    col for col in df_project.columns
    if col not in analysis_vars
]

print("\nNumber of columns not included in final analysis:")
print(len(excluded_from_final))

print("\nColumns not included in final analysis:")
print(excluded_from_final)


Number of columns not included in final analysis:
10

Columns not included in final analysis:
['i10_health', 'i11_health', 'age', 'household_size', 'household_children', 'i3_health', 'i4_health', 'qweek', 'feasibility_score', 'willingness_score']


In [13]:
# 2. Check missingness in final analysis variables

missing_summary = (
    df_project[analysis_vars]
    .isna()
    .sum()
    .to_frame("missing_n")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_n"]
    / len(df_project)
    * 100
)

missing_summary = missing_summary.sort_values(
    "missing_pct",
    ascending=False
)

print("\nMissingness summary:")
print(missing_summary.round(2))


Missingness summary:
                          missing_n  missing_pct
household_children_group        639         4.40
household_size_num              503         3.47
personal_test_status            418         2.88
household_test_status           416         2.87
covid_symptom_status            227         1.56
comorbidity_status               16         0.11
feasibility_binary                0         0.00
employment_status_4               0         0.00
employment_status_1               0         0.00
employment_status_2               0         0.00
employment_status_3               0         0.00
employment_status_7               0         0.00
employment_status_5               0         0.00
employment_status_6               0         0.00
gender                            0         0.00
i1_health                         0         0.00
i2_health                         0         0.00
i7a_health                        0         0.00
age_num                           0         0.0

In [14]:
# 3. Create complete-case analysis dataset

df_before_complete_case = df_project[analysis_vars].copy()

df_complete = df_before_complete_case.dropna().copy()

print("\nBefore complete-case deletion:", df_before_complete_case.shape)
print("After complete-case deletion:", df_complete.shape)

print(
    "Retained percentage:",
    round(
        len(df_complete) / len(df_before_complete_case) * 100,
        2
    ),
    "%"
)


Before complete-case deletion: (14512, 39)
After complete-case deletion: (13114, 39)
Retained percentage: 90.37 %


In [15]:
# 4. Compare continuous variables before vs after

continuous_vars = [
    "age_num",
    "household_size_num",
    "i1_health",
    "i2_health",
    "i7a_health"
]

continuous_comparison = pd.DataFrame({
    "Before_mean": df_before_complete_case[continuous_vars].mean(),
    "After_mean": df_complete[continuous_vars].mean(),
    "Before_median": df_before_complete_case[continuous_vars].median(),
    "After_median": df_complete[continuous_vars].median(),
    "Before_sd": df_before_complete_case[continuous_vars].std(),
    "After_sd": df_complete[continuous_vars].std()
})

print("\nContinuous-variable comparison:")
print(continuous_comparison.round(2))


Continuous-variable comparison:
                    Before_mean  After_mean  Before_median  After_median  \
age_num                   32.68       33.02           30.0          31.0   
household_size_num         3.79        3.77            4.0           4.0   
i1_health                  2.98        2.82            2.0           2.0   
i2_health                 15.01       14.97            4.0           4.0   
i7a_health                 1.89        1.84            1.0           1.0   

                    Before_sd  After_sd  
age_num                 10.93     11.02  
household_size_num       1.33      1.31  
i1_health               12.28      3.14  
i2_health               68.84     67.57  
i7a_health               4.03      3.45  


In [16]:
# 5. Compare categorical-variable proportions

categorical_comparison_vars = [
    "gender",
    "household_children_group",
    "personal_test_status",
    "household_test_status",
    "covid_symptom_status",
    "comorbidity_status",
    "survey_wave",
    "feasibility_binary",
    "willingness_binary"
]

for col in categorical_comparison_vars:
    print(f"\n{col}: before vs after complete-case deletion")

    comparison = pd.concat(
        [
            df_before_complete_case[col]
            .value_counts(normalize=True, dropna=False)
            .mul(100)
            .rename("Before(%)"),

            df_complete[col]
            .value_counts(normalize=True, dropna=False)
            .mul(100)
            .rename("After(%)")
        ],
        axis=1
    ).fillna(0)

    print(comparison.round(2))


gender: before vs after complete-case deletion
        Before(%)  After(%)
gender                     
Male         55.8     55.35
Female       44.2     44.65

household_children_group: before vs after complete-case deletion
                          Before(%)  After(%)
household_children_group                     
1                             42.01     44.46
0                             38.06     39.63
2_or_more                     15.53     15.91
NaN                            4.40      0.00

personal_test_status: before vs after complete-case deletion
                      Before(%)  After(%)
personal_test_status                     
not_tested                86.16     88.93
negative                   8.27      8.46
NaN                        2.88      0.00
pending                    1.87      1.80
positive                   0.83      0.81

household_test_status: before vs after complete-case deletion
                       Before(%)  After(%)
household_test_status               

In [17]:
# 6. Compare employment-status proportions

employment_comparison = pd.DataFrame({
    "Before_yes_pct": (
        df_before_complete_case[employment_vars]
        .mean()
        .mul(100)
    ),
    "After_yes_pct": (
        df_complete[employment_vars]
        .mean()
        .mul(100)
    )
})

print("\nEmployment-status comparison:")
print(employment_comparison.round(2))


Employment-status comparison:
                     Before_yes_pct  After_yes_pct
employment_status_1           64.89          66.22
employment_status_2           10.88          10.63
employment_status_3           18.37          18.32
employment_status_4            3.23           3.23
employment_status_5            5.08           4.60
employment_status_6            3.30           3.13
employment_status_7            2.02           1.43


In [18]:
# 7. Compare protective-behaviour response distributions

for col in behaviour_vars:
    print(f"\n{col}: before vs after complete-case deletion")

    behaviour_comparison = pd.concat(
        [
            df_before_complete_case[col]
            .value_counts(normalize=True, dropna=False)
            .mul(100)
            .rename("Before(%)"),

            df_complete[col]
            .value_counts(normalize=True, dropna=False)
            .mul(100)
            .rename("After(%)")
        ],
        axis=1
    ).fillna(0)

    print(behaviour_comparison.round(2))


i12_health_1: before vs after complete-case deletion
              Before(%)  After(%)
i12_health_1                     
Always            68.31     68.76
Frequently        20.06     19.96
Sometimes          7.86      7.72
Rarely             2.25      2.19
Not at all         1.52      1.36

i12_health_2: before vs after complete-case deletion
              Before(%)  After(%)
i12_health_2                     
Always            37.58     37.72
Frequently        29.62     29.95
Sometimes         18.60     18.32
Rarely             7.35      7.39
Not at all         6.85      6.63

i12_health_3: before vs after complete-case deletion
              Before(%)  After(%)
i12_health_3                     
Always            39.64     39.45
Frequently        29.22     29.45
Sometimes         19.02     19.03
Rarely             7.53      7.47
Not at all         4.58      4.61

i12_health_4: before vs after complete-case deletion
              Before(%)  After(%)
i12_health_4                     
Al

In [19]:
# 8. Dummy encode categorical predictors

categorical_cols = [
    *behaviour_vars,
    "gender",
    "household_children_group",
    "personal_test_status",
    "household_test_status",
    "covid_symptom_status",
    "comorbidity_status",
    "survey_wave"
]

df_model_ready = pd.get_dummies(
    df_complete,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print("\nModel-ready shape:", df_model_ready.shape)
print(
    "Remaining missing values:",
    df_model_ready.isna().sum().sum()
)


Model-ready shape: (13114, 113)
Remaining missing values: 0


In [20]:
# 9. Save final model-ready dataset

df_model_ready.to_csv(
    "../data/data_processed/df_model_ready.csv",
    index=False
)